# A working critic, the noise floor, and whether a search is admissible

`05` closed on two numbers that cannot both be signal. `ABLATE + entropy_coef=0.005`
reached held-out `cause_acc` **0.410** at seed 0 and **0.300** at seed 1 — same config, same
data, same code — while the `entropy_coef` sweep separated its two best points by **0.005**.

This notebook does three things in the order they can actually be settled.

### Q7 — does the critic work?

It does not, today: `value_ev` is **negative** in every run that has the entropy bonus
(−0.036 / −0.327 / −0.103). A baseline worse than the mean of the returns is not a critic,
and the series has been calling itself actor-critic PPO throughout.

This is first, and it is first for a specific reason: **`value_ev` is a training metric, not
a held-out one.** It is computed per batch from `metrics.jsonl` and averaged over 200 steps,
so it is not subject to the seed spread that governs `cause_acc` on 200 dev cases. A move
from −0.036 to +0.5 is unmistakable at one seed. Nothing needs to be measured first.

### Q8 — what is the noise floor, and did the fix move it?

Three seeds of the fixed configuration give σ of held-out `cause_acc`, and with it the
minimum difference this design can resolve at all. Seeds 0 and 1 also pair against the
unfixed runs already in `runs/`.

The prediction is deliberately modest, and there is a result below that sharpens it: with
`gamma = lam = 1.0` the critic is only a baseline, so a working one reduces the variance of
the gradient estimate without changing its expectation.

### Q9 — is a search admissible?

Only if the resolvable difference drops below the effects worth searching for. The harness
is built either way; Q7 and Q8 decide whether its selection layer may run.

In [ ]:
import sys, json, math, collections, subprocess, statistics as st
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "ppo_ac.py").exists() else Path("experiments/notebooks/smoke_test")
sys.path.insert(0, str(HERE.resolve()))

import torch
import ppo_ac
from ppo_ac import Config, train, resolve_device

MIN_FREE_MIB = 11000
BLOCKING_UNITS = ["vllm-qwen", "membraneclaw-agent"]
assert torch.cuda.is_available(), "no CUDA device; this notebook is the GPU half of the set"
smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader,nounits"],
    capture_output=True, text=True,
)
used, total = (int(x) for x in smi.stdout.strip().split(", "))
free = total - used
print(f"{torch.cuda.get_device_name(0)}: {free} / {total} MiB free per nvidia-smi")
running = [u for u in BLOCKING_UNITS
           if subprocess.run(["systemctl", "is-active", u], capture_output=True, text=True)
           .stdout.strip() == "active"]
assert not running, (
    f"{', '.join(running)} still up. A person with sudo has to stop them:\n"
    f"  sudo systemctl stop {' '.join(BLOCKING_UNITS)}"
)
assert free >= MIN_FREE_MIB, f"only {free} MiB free, need {MIN_FREE_MIB}"
print("card is free.")

## Why the critic is broken, and the three changes

### The batch shows the critic eight numbers

`gamma = lam = 1.0`, and `terminal_rewards` puts the reward on the last active token only.
So the return is **constant along a sequence**:

$$\text{returns}[b, t] = R_b \quad \text{for every active } t.$$

A batch of `prompts_per_step=8, samples_per_prompt=1` therefore presents exactly **eight
distinct target values**, replicated across ~770 masked positions (completions average 96.6
tokens). `critic_fit` pools tokens, so the token axis inflates $n$ without adding target
variance. A regressor shown eight numbers per step, whose mean moves every step, converges
to the running mean — which is what the metrics say happened:

| run | `value_mean` | `return_mean` | `value_std` | `value_ev` |
| --- | --- | --- | --- | --- |
| `ABLATE+ent` | 0.322 | 0.320 | 0.036 | **−0.036** |
| `ENT` | 0.340 | 0.341 | 0.050 | **−0.327** |
| `NODETACH+ent` | 0.372 | 0.364 | 0.042 | **−0.103** |

Three decimals of agreement, and a spread of 0.04. That is a constant.

>  **Superseded by `07_the_critic_cannot_work.ipynb`.** This section argued that the
>  representation was not the constraint, on the strength of `ValueHead`'s docstring
>  claiming `value_ev` +0.993 offline. That number has no held-out split: 25 batches of 8
>  is 200 independent targets against 2049 parameters, which interpolates. Held out by
>  sequence the same fit gives **+0.031**, and every intervention below was tried on a
>  premise that does not survive the split. `07` has the measurement and the verdict; the
>  runs here stand as the record of what was tried.

Kept for the record: Andrychowicz et al. 2020 (C47) finds separate policy and
value networks beat shared ones by enough that they deleted the shared variant and re-ran
the study, which independently argues against `--no-value-detach`.

### The three changes, all applied, all defaulting to the old behaviour

| `Config` field | default | this notebook | what it does |
| --- | --- | --- | --- |
| `critic_window` | `1` | **25** | fits the head over the last *K* batches instead of one. Active positions only, flattened to `(n_active, hidden)`, so batches of different completion length concatenate without padding — ~3 MiB per batch at the observed 96-token mean, 21 MiB at the 640 cap, held on CPU and moved to the device once per step rather than once per critic epoch. `K=25` is the offline probe's condition: 200 distinct targets rather than 8. |
| `recompute_advantages` | `False` | **True** | `advantages = (returns - fitted_values) * mask` after the critic loop. Until now `fitted_values` was computed and used **only for metrics**; the actor trained on advantages built from `old_values`, so the critic's eight dedicated steps did nothing for the batch that paid for them. C5 is the one improvement the paper proposes itself — stale advantages hurt, and recomputing per pass beat every variant they tried. One line, because $\lambda = 1$ makes `returns` the Monte-Carlo return, independent of $V$. |
| `value_clip_eps` | `0.2` | **`None`** | C13: PPO-style value-loss clipping hurt at every threshold they tested. It is **not** our bottleneck — measured `value_clip_frac` is 0.005–0.024 and `nodetach_ent` clipped nothing while still posting −0.103 — so this is a free cleanup, not the fix. |

**A trap worth recording: `value_clip_eps=0.0` is not "off".** It clamps the critic's
movement to nothing, which is strictly worse than clipping at 0.2. Since the generated CLI
types every flag from its default there is no way to pass `None`, so a negative value is the
sentinel — the same convention `value_init_bias` already uses, and `0.0` keeps its literal
meaning.

`lam` stays at 1.0. The paper recommends 0.9 (C8), but its environments have dense rewards
and 1000-step episodes; ours has one terminal reward per sequence, and lowering $\lambda$
means bootstrapping through a critic whose explained variance is negative. $\lambda$ becomes
a real knob the day `value_ev` is positive.

In [ ]:
BASE = dict(
    prompts_per_step=8, samples_per_prompt=1, max_new_tokens=640, temperature=1.0,
    lr=1e-5, lam=1.0, gamma=1.0, clip_eps=0.2, value_coef=0.5,
    whiten_advantages=False, lora_r=16, dtype="bfloat16", split="train",
    inner_epochs=4, value_detach=True, weights="ABLATE", entropy_coef=0.005,
)
CRITIC_FIX = dict(BASE, critic_window=25, recompute_advantages=True, value_clip_eps=None)

FIX_SEEDS = {s: f"ppo-qwen3-17b-ablate-ent-fix-s{s}" for s in (0, 1, 2)}
for seed, name in FIX_SEEDS.items():
    out = HERE / "runs" / name
    if (out / "metrics.jsonl").exists():
        print(f"{name}: exists, skipping (delete to retrain)")
        continue
    print(f"\n{'=' * 70}\n  critic fix: seed {seed} -> {name}\n{'=' * 70}")
    train(Config(**{**CRITIC_FIX, "seed": seed}), out, resolve_device("auto"))
    torch.cuda.empty_cache()
print("\nall three critic-fix runs present.")

## Q7 — did the critic start working?

Seed 0 of the fix pairs against `ppo-qwen3-17b-ablate-ent-s0`: same seed, same weights, same
entropy coefficient, same everything except the three fields above.

What to read, in order of how much it settles:

* **`value_ev` (out of sample)** — the gate. Positive and sustained; `> 0.3` is the bar.
* **`value_ev_fit` (in sample)** — should move first and by more. If the head still cannot
  fit the window it was just trained on, the window is not the constraint and the next
  suspect is target *scale*: C66 reports value-function normalisation mattering strongly and
  attributes it to "changing the speed of the value function fitting", and our within-batch
  return spread is `adv_std` 0.035–0.047 against a head whose single step is bounded near
  $\lVert h \rVert_1 \cdot \text{lr} \approx 0.02$. That is the opposite end from
  `normalize_value`, which normalises the head's *input* and was already measured slightly
  worse (+0.947 against +0.993).
* **`undefined` steps** — batches where every sequence drew nearly the same reward, so
  explained variance has no denominator. 28/200 on `MAIN`. The window should reduce this
  mechanically, since the denominator is now the window's variance, not one batch's.
* **`value_std`** — 0.04 is the constant-critic signature. It has to grow.

In [ ]:
def critic_summary(run_dir):
    rows = [json.loads(l) for l in open(HERE / "runs" / run_dir / "metrics.jsonl")]
    take = lambda k: [r[k] for r in rows if r.get(k) is not None]
    ev, fit = take("value_ev"), take("value_ev_fit")
    return {
        "value_ev": st.mean(ev) if ev else float("nan"),
        "value_ev_fit": st.mean(fit) if fit else float("nan"),
        "value_std": st.mean(take("value_std")),
        "value_mean": st.mean(take("value_mean")),
        "return_mean": st.mean(take("return_mean")),
        "undefined": len(rows) - len(ev),
        "clip_frac": st.mean(take("clip_frac")),
    }

BEFORE = {0: "ppo-qwen3-17b-ablate-ent-s0", 1: "ppo-qwen3-17b-ablate-ent-s1"}

print(f"{'run':40} {'value_ev':>9} {'ev_fit':>8} {'v_std':>7} {'V-R':>8} {'undef':>6}")
for label, mapping in (("before", BEFORE), ("after", FIX_SEEDS)):
    for s, name in sorted(mapping.items()):
        c = critic_summary(name)
        print(f"{name:40} {c['value_ev']:+9.3f} {c['value_ev_fit']:+8.3f} "
              f"{c['value_std']:7.3f} {c['value_mean'] - c['return_mean']:+8.4f} {c['undefined']:6d}")

ev_after = st.mean([critic_summary(n)["value_ev"] for n in FIX_SEEDS.values()])
ev_before = st.mean([critic_summary(n)["value_ev"] for n in BEFORE.values()])
print(f"\nvalue_ev  before {ev_before:+.3f}  ->  after {ev_after:+.3f}")
print(f"Q7 gate (value_ev > 0.3): {'PASS' if ev_after > 0.3 else 'FAIL'}")

# CHECK 15 -- `value_mean - return_mean` is the constant-critic tell: a head that has
# learned only the mean of the returns tracks it to three decimals. It has to stop doing
# that, whatever value_ev says.
_before_gap = st.mean([abs(critic_summary(n)["value_mean"] - critic_summary(n)["return_mean"])
                       for n in BEFORE.values()])
_after_gap = st.mean([abs(critic_summary(n)["value_mean"] - critic_summary(n)["return_mean"])
                      for n in FIX_SEEDS.values()])
print(f"\nCHECK 15 | |V - R| before {_before_gap:.4f}, after {_after_gap:.4f}")

## Q8 — the noise floor, and what it is actually made of

Three seeds of the fixed configuration give σ. Two things to read off it, and the second is
the one that matters for the search.

### The floor itself

For scale, from the paired McNemar tests already run on these same 200 cases:

| comparison | difference | result |
| --- | --- | --- |
| `frozen → ABLATE+ent` | 0.175 | p < 1e-4 |
| `ABLATE → PROBE` | 0.090 | **p = 0.095**, not significant |
| `entropy_coef` 0.005 vs 0.010 | 0.005 | never tested |

So ~0.09 is already at the edge of what 200 paired cases resolve, and the unfixed seed
spread was 0.11. The `05` sweep separated its two best points by twenty times less than that.

### The seed spread is not gradient noise — it is *which labels survive*

This is the result that decides what the critic fix can and cannot buy. Decomposing the two
unfixed seeds by the ceiling their own prediction histogram imposes:

| | labels used | ceiling | `cause_acc` | at % of ceiling |
| --- | --- | --- | --- | --- |
| seed 0 | 3 (scaling 79, biofouling 72, colloidal 48) | 0.440 | 0.410 | **93%** |
| seed 1 | 2 (biofouling 118, scaling 78) | 0.305 | 0.300 | **98%** |

```
seed-to-seed cause_acc gap: +0.110
seed-to-seed ceiling  gap: +0.135   <- larger than the accuracy gap
```

Both seeds are saturated against their own ceilings. Both are guessing from a shortened
list; they drew different lists. The 0.11 spread is therefore a **discrete, path-dependent
collapse event**, not gradient variance — and a baseline, which is all the critic is at
$\gamma = \lambda = 1$, reduces the variance of the gradient estimate without choosing which
basin the policy falls into.

The honest consequence: **the critic fix is a precondition for this being actor-critic PPO,
not a precondition for the search.** Those are different justifications. If σ does not move
below, that is the prediction confirming, not the fix failing.

In [ ]:
def final_eval(run_dir):
    rows = [json.loads(l) for l in open(HERE / "runs" / run_dir / "eval.jsonl")]
    return rows[-1]

fix_accs = [final_eval(n)["cause_acc"] for n in FIX_SEEDS.values()]
before_accs = [final_eval(n)["cause_acc"] for n in BEFORE.values()]

print(f"{'run':40} {'cause_acc':>10} {'flags_acc':>10} {'numeric':>8} {'reward':>8}")
for label, mapping in (("before", BEFORE), ("after", FIX_SEEDS)):
    for s, name in sorted(mapping.items()):
        r = final_eval(name)
        print(f"{name:40} {r['cause_acc']:10.3f} {r['flags_acc']:10.3f} "
              f"{r['numeric_acc']:8.3f} {r['reward']:8.3f}")

sigma = st.stdev(fix_accs)
print(f"\nafter : mean {st.mean(fix_accs):.3f}  sigma {sigma:.3f}  spread {max(fix_accs)-min(fix_accs):.3f}")
print(f"before: {before_accs} (n=2, spread {max(before_accs)-min(before_accs):.3f})")

mde_1 = 2 * math.sqrt(2) * sigma
mde_3 = 2 * math.sqrt(2) * sigma / math.sqrt(3)
print(f"\nminimum resolvable difference, 1 seed per config:  {mde_1:.3f}")
print(f"minimum resolvable difference, 3 seeds per config: {mde_3:.3f}")

# Did the task move at all? Stated separately from the critic, because they are
# separate claims and only one of them is what the fix was for.
print("\nnumeric_acc and exact_match across the fixed runs -- the arithmetic has never moved:")
for s, name in sorted(FIX_SEEDS.items()):
    rows = [json.loads(l) for l in open(HERE / "runs" / name / "eval.jsonl")]
    print(f"  seed {s}: numeric {[round(r['numeric_acc'], 3) for r in rows]}")

## The gate that has to exist before any search runs

`04` and `05` scored configurations on held-out `cause_acc` and nothing else, and the table
in Q8 is what that metric cannot see. `eval.py` already stores `predicted_cause_hist`;
nothing derives from it. With the seven causes balanced to within one case per split, a
policy that only emits some of them cannot score above

$$\text{ceiling} = \frac{1}{N}\sum_k \min\!\big(\text{predicted}_k,\ \text{true}_k\big).$$

The thresholds below are anchored to the **frozen policy** rather than chosen: a trained
policy that emits fewer distinct causes than the model it started from has lost something,
whatever its accuracy says.

Two consequences, both load-bearing for Q9:

* A search scored on `cause_acc` alone walks straight into this. Emitting the two most
  frequent causes is locally excellent and globally degenerate.
* Coverage cannot only be a hard gate. If it prunes every candidate the search has nothing
  to rank — so it is reported as a second objective as well as enforced as a constraint.

In [ ]:
DEV = Path("/home/bayan/MembraneClaw/experiments/membrane_grpo/data/dev.jsonl")
TRUE_CAUSES = collections.Counter(json.loads(l)["answer"]["root_cause"] for l in open(DEV))
N_CAUSES = len(TRUE_CAUSES)

def coverage_metrics(hist, true_counts=TRUE_CAUSES):
    """Label-space health of one greedy evaluation, from `predicted_cause_hist`.

    `acc_ceiling` is the best `cause_acc` this histogram admits: pair each predicted label
    with the true cases carrying it, and no policy beats the overlap. A run sitting at its
    own ceiling is not discriminating, it is guessing from a shortened list.
    """
    total = sum(hist.values()) or 1
    p = [v / total for v in hist.values() if v > 0]
    return {
        "labels_used": sum(1 for v in hist.values() if v > 0),
        "label_coverage": sum(1 for v in hist.values() if v > 0) / N_CAUSES,
        "pred_entropy": -sum(q * math.log(q) for q in p) / math.log(N_CAUSES),
        "top2_share": sum(sorted(hist.values(), reverse=True)[:2]) / total,
        "acc_ceiling": sum(min(v, true_counts.get(k, 0)) for k, v in hist.items()) / total,
    }

PAIRED = HERE / "runs" / "paired"
FROZEN = coverage_metrics(json.load(open(PAIRED / "base.json"))["overall"]["predicted_cause_hist"])
print(f"frozen policy anchors the gate: {FROZEN['labels_used']}/{N_CAUSES} labels, "
      f"H={FROZEN['pred_entropy']:.2f}, top2={FROZEN['top2_share']:.2f}\n")

def gate(summary, frozen=FROZEN):
    """Hard constraints. A violating trial is pruned, not scored."""
    m = coverage_metrics(summary["predicted_cause_hist"])
    return m, {
        "coverage_not_below_frozen": m["labels_used"] >= frozen["labels_used"],
        "entropy_not_below_frozen": m["pred_entropy"] >= frozen["pred_entropy"],
        "not_saturated": summary["cause_acc"] < 0.95 * m["acc_ceiling"],
        "schema_floor": summary["schema_ok"] >= 0.95,
        "length_sane": summary.get("completion_tokens_mean", 0) <= 300,
    }

ORDER = ["base", "main", "ablate", "probe", "ie2", "nodetach",
         "ent", "ablate_ent", "probe_ent", "nodetach_ent", "ablate_ent_s1"]
print(f"{'run':15} {'acc':>5} {'ceil':>5} {'used':>5} {'H':>5} {'top2':>5}  verdict / first failure")
for name in ORDER:
    s = json.load(open(PAIRED / f"{name}.json"))["overall"]
    m, checks = gate(s)
    bad = [k for k, ok in checks.items() if not ok]
    print(f"{name:15} {s['cause_acc']:5.3f} {m['acc_ceiling']:5.3f} {m['labels_used']:5d} "
          f"{m['pred_entropy']:5.2f} {m['top2_share']:5.2f}  {'PASS' if not bad else f'PRUNE ({bad[0]})'}")

# CHECK 16 -- the gate must be able to reject the series' own headline, or it is decoration.
_s = json.load(open(PAIRED / "ablate_ent.json"))["overall"]
_m, _c = gate(_s)
assert not all(_c.values()), "gate passes the collapsed headline run -- it is not a gate"
print(f"\nCHECK 16 ok | the 0.410 headline is pruned: ceiling {_m['acc_ceiling']:.3f}, "
      f"{_m['labels_used']}/{N_CAUSES} labels used")
print("only the frozen policy and PROBE survive: every trained run is more collapsed than "
      "the model it started from.")

## Q9 — the search, and whether it may run

The harness is complete. Its selection layer is not enabled, and the condition is an
assertion rather than a judgement call:

> the minimum difference three seeds can resolve must be smaller than the smallest effect
> worth acting on.

Take that threshold as **0.05** — half the `ABLATE → PROBE` difference that already failed
to reach significance on these 200 cases.

### Why successive halving fits here unusually well

A naive grid over `clip_eps` (0.05 / 0.1 / 0.2), `inner_epochs` (2 / 4 / 10) and
`entropy_coef` (0.002 / 0.005 / 0.010) is 27 configurations; at three seeds that is 81 hours
on a card that is also a production service. Three properties make pruning effective here:

* `eval_every=25` already writes held-out scores **during** the run, so intermediate scores
  exist without adding anything.
* Degenerate configurations declare themselves early — `entropy_coef=0.020` blew up
  completion length almost immediately, `MAIN`'s collapse was visible by step 100–125.
  Killing at step 50 costs ~15 minutes against ~60 (measured: 14.5 s/step).
* Bad configurations are **slower** — `c020` ran roughly twice as long because completions
  hit the 640-token cap — so pruning them recovers more than their share of the budget.

### Why the objective is not the reward

Reward is not comparable across weight sets, and the series has been burned by this already:
`PROBE` reaches 0.518 against `MAIN`'s 0.271 while collecting a third of it from `format`
before the first gradient step, because schema validity starts at 0.965. Only accuracies
transfer. The search optimises `cause_acc` subject to the gate, and reports coverage
alongside rather than folding it into a scalar.

### What is still missing for an unattended search

`train()` runs to completion; there is no hook to stop it at step 50. Pruning needs either a
callback at each `eval_every` boundary that can raise, or the driver launching `ppo_ac.py`
through its generated CLI and killing the subprocess. The CLI already exposes every `Config`
field, so the subprocess route needs no module changes — at the cost of the in-process
convention every other notebook here uses. That choice should be made after Q8 reports.

In [ ]:
SEARCH_SPACE = {
    "clip_eps":     [0.05, 0.1, 0.2],
    "inner_epochs": [2, 4, 10],
    "entropy_coef": [0.002, 0.005, 0.010],
}
MDE_THRESHOLD = 0.05
SEEDS_PER_CONFIG = 3

def trial_result(run_dir, summary):
    """One trial's record: objective, constraints, and why it was pruned."""
    m, checks = gate(summary)
    return {
        "run": run_dir,
        "objective": summary["cause_acc"],      # weight-set invariant, unlike reward
        "coverage": m["label_coverage"],
        "acc_ceiling": m["acc_ceiling"],
        "gates": checks,
        "pruned": [k for k, ok in checks.items() if not ok],
    }

mde = 2 * math.sqrt(2) * sigma / math.sqrt(SEEDS_PER_CONFIG)
print(f"sigma = {sigma:.3f} -> resolvable at {SEEDS_PER_CONFIG} seeds = {mde:.3f}")
print(f"threshold = {MDE_THRESHOLD}")
print(f"\nQ9: the search is {'ADMISSIBLE' if mde <= MDE_THRESHOLD else 'NOT admissible'}.")

# The selection layer stays off until the measurement says otherwise. A search that cannot
# resolve its own effects still returns a winner, and that winner is a seed.
assert mde <= MDE_THRESHOLD, (
    f"sigma {sigma:.3f} resolves only {mde:.3f}; the smallest effect worth acting on is "
    f"{MDE_THRESHOLD}. Raise seeds per config, enlarge the evaluation split, or attack the "
    f"collapse that Q8 shows is producing the spread."
)
print("\nsearch space:", json.dumps(SEARCH_SPACE))

---

## Appendix — can this model do the arithmetic at all?

Small, and here rather than in a notebook of its own because it decides how to read every
`numeric_acc` in the tables above.

`exact_match` is **0.000** at all 27 evaluation points across all 16 runs. `numeric_acc`
peaks at 0.108. 168 of 200 cases get all three numbers wrong. And the `easy` tier — where
the feed temperatures are equal, `TCF` cancels, and the first computation is a plain percent
change — scores the same as `hard` (0.403 against 0.423). The model is not failing the
temperature correction; it is not doing the arithmetic at all.

Two readings, and they lead opposite ways:

* **The capability is absent.** Then no reward shapes it into existence: a sparse scalar
  cannot teach a multi-step computation that never appears in a sample, because there is
  nothing partial to sharpen. This is the reason the README gives for rejecting
  Qwen2.5-0.5B — its `cause_acc` "sat *exactly* at chance, leaving RL nothing partial to
  sharpen". For the numeric component, the 1.7B is in that position.
* **The capability is present but not elicited.** Then the fix is upstream of RL entirely:
  a prompt that shows the working, or an SFT cold start that establishes the format.

Few-shot prompting separates them for one greedy pass and no training. If two worked
examples move `numeric_acc` off the floor, the arithmetic is in there and the RL setup was
never asking for it. If they do not, SFT would teach the *shape* of a calculation the model
cannot carry out, and the honest options become a calculator tool or a larger model.

**This needs the card** — Qwen3-1.7B is 3.4 GiB of bf16 weights and a training run leaves
about 1.7 GiB free — so it queues behind the runs above rather than running alongside them.

### Result: the capability is absent

Run on the frozen policy, greedy, 60 dev cases, three conditions:

| | `numeric_acc` | **all three right** | `exact_match` | `schema_ok` | `cause_acc` |
| --- | --- | --- | --- | --- | --- |
| zero-shot | 0.033 | **0.000** | 0.000 | 0.950 | 0.317 |
| 2-shot, answer JSON only | 0.072 | **0.000** | 0.000 | 1.000 | 0.067 |
| 2-shot, **showing the arithmetic** | 0.033 | **0.000** | 0.000 | 1.000 | 0.067 |

Not one case in sixty gets all three numbers right, in any condition. Showing the working
explicitly — the strongest elicitation available short of training — moves `numeric_acc` by
exactly nothing (0.033 → 0.033), and the weaker prompt's apparent gain to 0.072 arrives
alongside `schema_ok` going 0.950 → 1.000, so it is format compliance rather than
computation.

That is what an absent capability looks like. Few-shot prompting is the standard way to
elicit a latent one, and three attempts elicited nothing. A sparse scalar reward cannot do
better: it can only reweight behaviour that already appears in a sample, and this behaviour
never appears.

A second reading falls out of the same table. Few-shot **collapses** `cause_acc` from 0.317
to 0.067 — worse than the 1/7 chance rate. The model is copying the demonstrated answers
rather than reasoning from the record, which is its own evidence about what it is doing on
this task.

**So the two failure modes are different in kind**, and only one of them is about the model:

* **The arithmetic** is a capability the model does not have. No amount of RL creates it;
  the options are a larger model or a calculator tool.
* **The label collapse** is not. The frozen policy emits 5 of 7 causes at entropy 0.61;
  every trained run is narrower (3–4 causes, 0.55–0.57). Training destroyed diversity that
  was there to begin with, and that is a training-dynamics problem with training-dynamics
  fixes — a KL penalty to the frozen policy, a coverage term in the reward, or an SFT cold
  start.

In [ ]:
# Ran as `fewshot.py` / `fewshot_cot.py`; the numbers are in the table above.
# The shots come from `train`, never from `dev`, and are injected between the
# system message and the case so the system prompt still leads.
N_PROBE, WORKED_EXAMPLES = 60, 2
print("zero-shot           numeric_acc 0.033   all three right 0.000")
print("2-shot, answer only numeric_acc 0.072   all three right 0.000")
print("2-shot, with working numeric_acc 0.033  all three right 0.000")